# VTC BETTER-data lift -- STAGED / resumable

This is the production version of the VTC better-data Colab run. It is chunked into restartable stages and writes artifacts under `/content/vtc_stage` so a Colab disconnect does not erase the whole run.

Stages:
1. install + runtime check
2. repo/corpus/split
3. model + LoRA adapter load
4. base eval -> `base.json`
5. train/resume -> adapter + checkpoints
6. trained eval -> `trained.json`
7. final report -> `report.json` + `NET LIFT`


## 0. Stage paths and helpers


In [ ]:

import json, os, time
from pathlib import Path
STAGE = Path('/content/vtc_stage')
STAGE.mkdir(parents=True, exist_ok=True)

def save_json(name, obj):
    path = STAGE / name
    path.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding='utf-8')
    print('SAVED', path)
    return str(path)

def load_json(name):
    return json.loads((STAGE / name).read_text(encoding='utf-8'))

def mark(name, **data):
    payload = {'stage': name, 'ts': time.time(), **data}
    return save_json(f'{name}.stage.json', payload)

print('STAGE_DIR', STAGE)


## 1. Install (fast -- no unsloth, no source builds)


In [ ]:
!pip install -q -U peft bitsandbytes accelerate datasets 'transformers>=4.44'
import torch
print('CUDA available:', torch.cuda.is_available())  # must be True (Runtime>Change runtime type>T4)


## 2. Repo + corpus + honest split


In [ ]:

import os, sys
SCBE_REF = os.environ.get('SCBE_REF', 'feat/vtc-better-colab-runnable')
!rm -rf /content/scbe
!git clone --depth 1 --branch $SCBE_REF https://github.com/issdandavis/SCBE-AETHERMOORE.git /content/scbe
sys.path.insert(0, '/content/scbe')
MBPP_PATH = '/content/scbe/training-data/sft/vtc_mbpp_refined.jsonl'
assert os.path.exists(MBPP_PATH), f'missing corpus: {MBPP_PATH}'

from python.helm.better_corpus import build as build_better
from python.helm import public_bench as pb
from python.helm.vtc_split import load_corpus, split_by_task_id, write_train_sft

CORPUS_PATH = '/content/vtc_better.jsonl'
TRAIN_SFT = '/content/train.sft.jsonl'
PUBLIC_K = int(os.environ.get('PUBLIC_K', '1'))
EVAL_LIMIT = int(os.environ.get('EVAL_LIMIT', '50'))
EVAL_KIND = os.environ.get('EVAL_KIND', 'pitfall').strip().lower()

info = build_better(MBPP_PATH, CORPUS_PATH)
records = load_corpus(CORPUS_PATH)
problems = pb.pull_mbpp()
split = split_by_task_id(records, problems, public_k=PUBLIC_K)
write_train_sft(split['train_records'], TRAIN_SFT)
if EVAL_KIND == 'pitfall':
    from python.helm.pitfall_eval import eval_problems as _pitfall_eval_problems, discriminating
    eval_problems = _pitfall_eval_problems()
    assert len(eval_problems) == len(discriminating()), 'pitfall eval lost discriminating items'
else:
    eval_problems = split['eval_problems'][:EVAL_LIMIT] if EVAL_LIMIT else split['eval_problems']
    assert not (split['train_ids'] & {p['task_id'] for p in eval_problems}), 'train/eval LEAK'

save_json('data_info.json', {
    'better_corpus': info,
    'train_records': len(split['train_records']),
    'heldout_eval': len(eval_problems),
    'eval_kind': EVAL_KIND,
    'public_k': PUBLIC_K,
    'eval_limit': EVAL_LIMIT,
    'train_sft': TRAIN_SFT,
    'corpus_path': CORPUS_PATH,
})
mark('data_ready', train_records=len(split['train_records']), heldout_eval=len(eval_problems), eval_kind=EVAL_KIND)
print('better corpus:', info)
print('train records:', len(split['train_records']), ' eval kind:', EVAL_KIND, ' held-out eval:', len(eval_problems))


## 3. Model config


In [ ]:

BASE_MODEL = os.environ.get('BASE_MODEL', 'Qwen/Qwen2.5-Coder-3B-Instruct')
OUT = '/content/vtc-fast-run'
ADAPTER_OUT = '/content/vtc_stage/adapter'
MAXLEN = int(os.environ.get('MAXLEN', '2048'))
EPOCHS = float(os.environ.get('EPOCHS', '2'))
MAX_NEW_TOKENS = int(os.environ.get('MAX_NEW_TOKENS', '512'))
print({'BASE_MODEL': BASE_MODEL, 'OUT': OUT, 'ADAPTER_OUT': ADAPTER_OUT, 'MAXLEN': MAXLEN, 'EPOCHS': EPOCHS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS})
mark('config_ready', base_model=BASE_MODEL)


## 5. Load 4-bit model + LoRA adapter + generators


In [ ]:
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, device_map='auto')
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

def make_prompt(problem):
    public = '\n'.join(list(problem.get('test_list', []))[:PUBLIC_K])
    return ((problem.get('prompt') or problem.get('text') or '').strip()
            + '\n\nWrite a complete Python solution. It must make this example pass:\n'
            + public + '\nReturn ONLY the code.')

def extract_code(text):
    # robust: largest fenced block; else drop leading prose to the first code line. Handles
    # unclosed ``` (truncated gen) and bare code with no fences -- why base scored 0 before.
    blocks = re.findall(r'```(?:python)?\s*(.*?)```', text or '', re.S)
    if blocks: return max(blocks, key=len).strip()
    body = re.sub(r'^\s*```(?:python)?\s*', '', (text or '').strip())
    lines = body.splitlines()
    for i, ln in enumerate(lines):
        if ln.lstrip().startswith(('def ', 'import ', 'from ', 'class ', '@')):
            return '\n'.join(lines[i:]).strip()
    return body.strip()

def _gen(prompt):
    text = tokenizer.apply_chat_template([{'role':'user','content':prompt}], tokenize=False,
             add_generation_prompt=True)
    enc = tokenizer(text, return_tensors='pt').to(model.device)
    out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
             pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)

make_hf_generator = lambda: (lambda problem: extract_code(_gen(make_prompt(problem))))
make_hf_ask = lambda: _gen  # prompt -> str, for the repair loop

mark('model_loaded', base_model=BASE_MODEL)


## 5. Base eval only -> base.json


In [ ]:

from python.helm.code_lift import solve_rate
from python.helm import public_bench as _pb
model.eval()
gen = make_hf_generator()

for p in eval_problems[:2]:
    with model.disable_adapter():
        code = extract_code(_gen(make_prompt(p)))
    chk = _pb._verify(code, p['test_list'][:PUBLIC_K], [], p.get('test_imports', []))
    print('--- task', p['task_id'], '| public_passed:', chk['public_passed'], '---')
    print(code[:200].replace(chr(10), ' | '))

print('Evaluating BASE (adapter disabled)...')
with model.disable_adapter():
    base = solve_rate(eval_problems, gen, public_k=PUBLIC_K)
save_json('base.json', base)
mark('base_eval_done', solved=len(base.get('solved', [])), failed=len(base.get('failed', [])))
print('BASE saved:', STAGE / 'base.json')


## 6. Train/resume LoRA -> adapter checkpoint


In [ ]:

from pathlib import Path
from datasets import load_dataset
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
train_ds = load_dataset('json', data_files=TRAIN_SFT, split='train')

def tok(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return tokenizer(text, truncation=True, max_length=MAXLEN)

train_tok = train_ds.map(tok, remove_columns=train_ds.column_names)
args = TrainingArguments(
    output_dir=OUT,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=EPOCHS,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=5,
    bf16=True,
    optim='paged_adamw_8bit',
    report_to='none',
    save_strategy='steps',
    save_steps=5,
    save_total_limit=2,
)
trainer = Trainer(model=model, args=args, train_dataset=train_tok, data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False))
resume = True if list(Path(OUT).glob('checkpoint-*')) else None
train_output = trainer.train(resume_from_checkpoint=resume)
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
save_json('train_metrics.json', train_output.metrics)
mark('train_done', adapter_out=ADAPTER_OUT, metrics=train_output.metrics)
print('ADAPTER saved:', ADAPTER_OUT)


## 7. Trained eval + final report


In [ ]:

from python.helm.code_lift import lift_from_solve, render
print('Evaluating TRAINED (adapter enabled)...')
trained = solve_rate(eval_problems, gen, public_k=PUBLIC_K)
save_json('trained.json', trained)
base = load_json('base.json')
report = lift_from_solve(base, trained)
save_json('report.json', report)
print(); print(render(report)); print()
print('newly solved ids:', sorted(report['newly_solved']))
print('regressed ids   :', sorted(report['regressed']))
mark('report_done', net_lift=report.get('net_lift'), newly_solved=sorted(report.get('newly_solved', [])), regressed=sorted(report.get('regressed', [])))
print('REPORT_JSON:', STAGE / 'report.json')
